In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from config import PatientDir
from config.intervals import SEGMENT
from cycle_extraction.cycle_extraction_for_segments import handle_feature_outliers
from feature_extraction.extract_features import FeatureNames

In [ ]:
# Init constants
feat_names = FeatureNames.ALL_ORDERED.copy()
if 'corrcoef' in feat_names: feat_names.remove('corrcoef')

In [ ]:
# Plot original vs. outlier handled (clipped) features
def plot_original_vs_clipped(seg_feats, clipping_quantile_bounds: list[float]):
    for feat in feat_names:
        fig, axs = plt.subplots(nrows=1 + len(clipping_quantile_bounds), ncols=1, sharex=True, sharey=True,
                                figsize=(20, 3 * (1 + len(clipping_quantile_bounds))))
        orig = seg_feats[feat].values

        axs[0].set_title('Original')
        axs[0].plot(orig, lw=0.25, alpha=0.8, color='black')

        for bound_i, bound in enumerate(clipping_quantile_bounds):
            clipped = handle_feature_outliers(orig, bound)
            ax = axs[bound_i + 1]
            ax.set_title(f'Clipped with bound {bound}')
            ax.plot(clipped, lw=0.25, alpha=0.8)

        fig.tight_layout()
        fig.suptitle(feat, x=0.1)
        fig.show()

In [ ]:
# Load data
pdirs_checked = [
    PatientDir('/data/home/webb/d/UNEEG/datasets/competition/competition-1'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/competition/competition-2'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/competition/competition-3'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-01'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-03'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-04'), # <- Maybe this one needs a lower bound
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-05'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-07'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-12'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-15'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-16'),
    PatientDir('/data/home/webb/d/UNEEG/datasets/ultra2/U002-DE01-17') # <- Maybe this one needs a lower bound for acfw
]
pdirs_unchecked = [
]
pdir = pdirs_unchecked[0]
print(pdir.name)

szrs = pd.read_pickle(pdir.all_szr_starts_file.pickle)['start_mtz']
seg_feats = pd.read_pickle(pdir.filled_features_for_segs.pickle)
end_mtz = seg_feats['start_mtz'] + SEGMENT.exact_dur
seg_feats.insert(seg_feats.columns.get_loc("start_mtz") + 1, "end_mtz", end_mtz)
# seg_feats.head()

In [ ]:
plot_original_vs_clipped(seg_feats, clipping_quantile_bounds=[0.99999])